# Récupération et consolidation des données météorologiques

Le but de ce script est de procéder à la récupération des données météorologiques issues de Meteo France et de consolider ces données en un fichier paquet exploitable.

Les données de Météo France sont subdivisées en de nombreux fichiers, disponibles sur la page https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-quotidiennes/. (Pas de fichier unique téléchargeable, il serait trop imposante)
Avant de procéder à l'execution du script, et pour simplifier la récupération des données, il faut executer la commande suivante en ligne de commande:


(echo '"nom","url"'; curl "https://www.data.gouv.fr/api/1/datasets/donnees-climatologiques-de-base-quotidiennes/" | jq -r '.resources[] | [.title, .url] | @csv') > urls_stables.csv


Cette commande va réaliser une analyse préliminaire des données de la page et générer un fichier contenant les urls récupérées, ce fichier servira se base pour le reste des traitements effectués.

In [2]:
import pandas as pd
import re
from pathlib import Path

In [7]:
#Configuration
srcFolder = "../data/raw/data.gouv/"    #Répertoire source
dstFolder = "../data/processed/test/"    #Repertoire destination

In [4]:
#Analyse des données brutes, extraction de meta données depuis l'url, enregistrement dans un fichier consolidé

dfUrl = pd.read_csv(srcFolder+"urls_stables.csv")
dfUrl = dfUrl[dfUrl['nom'].str.contains('vent', case=False)].copy()

dfUrl["debut"] = 0
dfUrl["fin"] = 0

for index, row in dfUrl.iterrows():
    r = re.search(r'(\d{4})-(\d{4})', row["nom"])
    if r != None:
        dfUrl.loc[index, "debut"] = int(r.group(1))
        dfUrl.loc[index, "fin"] = int(r.group(2))

dfUrl.to_csv(srcFolder+"urls_stables_consolide.csv", index=False)

In [5]:
#On charge la liste des fichiers sources, on ne conserve que ceux ultérieurs à 1950
dfSources = pd.read_csv(srcFolder+"urls_stables_consolide.csv")
#On garde que ceux > 1950
dfSources = dfSources[(dfSources["fin"] > 1950)].copy()
dfSources


,nom,url,debut,fin
1,QUOT_departement_01_periode_1950-2023_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,1950,2023
2,QUOT_departement_01_periode_2024-2025_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,2024,2025
4,QUOT_departement_02_periode_1950-2023_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,1950,2023
5,QUOT_departement_02_periode_2024-2025_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,2024,2025
7,QUOT_departement_03_periode_1950-2023_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,1950,2023
...,...,...,...,...
308,QUOT_departement_987_periode_1950-2023_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,1950,2023
309,QUOT_departement_987_periode_2024-2025_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,2024,2025
311,QUOT_departement_988_periode_1950-2023_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,1950,2023
312,QUOT_departement_988_periode_2024-2025_RR-T-Vent,https://object.files.data.gouv.fr/meteofrance/...,2024,2025


In [ ]:
#Téléchargement des fichiers

fileIndex = 1
for index, row in dfSources.iterrows():
    print(f"Traitement du fichier {fileIndex} {row["nom"]}")
    fichier = Path(f"{srcFolder}{fileIndex}.csv")
    if fichier.exists():
        print("Fichier déjà existant, on continue")
        fileIndex = fileIndex +1
        continue
    
    df = pd.read_csv(row["url"], compression='gzip', sep=';')  
    df2 = df[["TX","TN","LAT","LON","AAAAMMJJ","ALTI"]]
    dff = df2[(df2["TX"] > 0) & (df2["TN"] > 0)]
    dff.to_csv(f"{srcFolder}{fileIndex}.csv")
    print("Saved !")
    fileIndex = fileIndex +1
    

Traitement du fichier 1 QUOT_departement_01_periode_1950-2023_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 2 QUOT_departement_01_periode_2024-2025_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 3 QUOT_departement_02_periode_1950-2023_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 4 QUOT_departement_02_periode_2024-2025_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 5 QUOT_departement_03_periode_1950-2023_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 6 QUOT_departement_03_periode_2024-2025_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 7 QUOT_departement_04_periode_1950-2023_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 8 QUOT_departement_04_periode_2024-2025_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 9 QUOT_departement_05_periode_1950-2023_RR-T-Vent
Fichier déjà existant, on continue
Traitement du fichier 10 QUOT_departe

In [ ]:
#Consolidation dans un fichier parquet
dataFrames = []

for i in range(fileIndex):
    i = i+1
    print(f"Traitement fichier {i}")
    df = pd.read_csv(f"{srcFolder}{i}.csv")
    dataFrames.append(df)

dfAll = pd.concat(dataFrames, ignore_index=True)
dfAll.to_parquet(f"{dstFolder}meteo.parquet", engine='fastparquet')

Traitement fichier 1
Traitement fichier 2
